# Week 6 Assignment: Apache Spark Architecture and Data Processing


## Objective:
## Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, and optimized file formats.

In [10]:
import os
os.environ['SPARK_HOME'] = "C://Program Files//Spark"
os.environ['PYSPARK_DRIVER_PYTHON'] = 'jupyter'
os.environ['PYSPARK_DRIVER_PYTHON_OPTS'] = 'lab'
os.environ['PYSPARK_PYTHON'] = 'python'
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [12]:
import pyspark
print(pyspark.__version__)

4.1.2


In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

In [16]:
spark = SparkSession.builder \
    .appName("Week6Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [18]:
spark

In [25]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Week6")
    .master("local[*]")
    .getOrCreate()
)



+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



# Read CSV File

## This step reads the input CSV file with header and automatically infers the schema.

In [26]:
df = spark.read.csv(
    "data/employees.csv",
    header=True,
    inferSchema=True
)

df.show()

+-------+----+--------+------+------+----------+-----------+------+-----------+------+
|   Name| Age|Category|Region|Salary|Experience|JoiningDate|Rating| Department|Remote|
+-------+----+--------+------+------+----------+-----------+------+-----------+------+
|  Alice|  25|      IT| North| 50000|         2| 2022-03-15|   4.2|Engineering|   Yes|
|    Bob|  30|      HR| South| 60000|         5| 2019-07-01|   3.8|Recruitment|    No|
|Charlie|  35|      IT|  East| 70000|        10| 2014-01-20|   4.7|Engineering|   Yes|
|  David|  28| Finance|  West| 55000|         4| 2020-06-10|  NULL| Accounting|    No|
|    Eva|  22|      IT| North| 45000|         1| 2023-09-01|   3.5|Engineering|   Yes|
|  Frank|  40|      HR| South| 80000|        15| 2009-04-22|   4.9|Recruitment|    No|
|  Grace|  29| Finance|  East| 62000|         6| 2018-11-05|   4.1| Accounting|   Yes|
|  Henry|  31|      IT|  West| 72000|         7| 2017-08-30|   4.4|Engineering|    No|
|    Ivy|  27|      HR| North| 58000|      

In [27]:
print("Schema")
df.printSchema()

Schema
root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- JoiningDate: date (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Department: string (nullable = true)
 |-- Remote: string (nullable = true)



# Missing Values Identified

In [28]:
df.filter(col("salary").isNull()).count()

2

In [29]:
df.filter(col("salary").isNull()).show()

+------+---+--------+------+------+----------+-----------+------+-----------+------+
|  Name|Age|Category|Region|Salary|Experience|JoiningDate|Rating| Department|Remote|
+------+---+--------+------+------+----------+-----------+------+-----------+------+
|Nathan| 38|      IT| South|  NULL|        12| 2012-06-15|   4.6|Engineering|    No|
|   Zoe| 39|      IT| South|  NULL|        14| 2010-09-14|   4.6|Engineering|    No|
+------+---+--------+------+------+----------+-----------+------+-----------+------+



In [30]:
df.filter(col("age").isNull()).count()

4

In [31]:
df.filter(col("age").isNull()).show()

+------+----+--------+------+------+----------+-----------+------+-----------+------+
|  Name| Age|Category|Region|Salary|Experience|JoiningDate|Rating| Department|Remote|
+------+----+--------+------+------+----------+-----------+------+-----------+------+
| Karen|NULL|      IT|  East| 53000|         3| 2021-07-19|   3.7|Engineering|   Yes|
|   Sam|NULL|      HR| North| 56000|         4| 2020-08-22|   3.8|Recruitment|   Yes|
|Carlos|NULL|      HR|  West| 65000|         8| 2016-10-09|   4.4|Recruitment|   Yes|
|  Mike|NULL| Finance| North| 82000|        15| 2009-08-18|   4.6| Accounting|   Yes|
+------+----+--------+------+------+----------+-----------+------+-----------+------+



# Handle Missing Values

In [32]:
df = df.fillna({"salary":0,"age":0})

In [33]:
df.filter(col("salary").isNull()).show()

+----+---+--------+------+------+----------+-----------+------+----------+------+
|Name|Age|Category|Region|Salary|Experience|JoiningDate|Rating|Department|Remote|
+----+---+--------+------+------+----------+-----------+------+----------+------+
+----+---+--------+------+------+----------+-----------+------+----------+------+



In [34]:
df.filter(col("age").isNull()).show()

+----+---+--------+------+------+----------+-----------+------+----------+------+
|Name|Age|Category|Region|Salary|Experience|JoiningDate|Rating|Department|Remote|
+----+---+--------+------+------+----------+-----------+------+----------+------+
+----+---+--------+------+------+----------+-----------+------+----------+------+



# Column Renamed 

In [35]:
df = df.withColumnRenamed("salary","monthly_salary")

In [36]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- monthly_salary: integer (nullable = false)
 |-- Experience: integer (nullable = true)
 |-- JoiningDate: date (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Department: string (nullable = true)
 |-- Remote: string (nullable = true)



# New Column Added

In [49]:
df = df.withColumn(
    "annual_salary",
    col("monthly_salary") * 12
)

In [51]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- monthly_salary: integer (nullable = false)
 |-- Experience: integer (nullable = true)
 |-- JoiningDate: date (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Department: string (nullable = true)
 |-- Remote: string (nullable = true)
 |-- annual_salary: integer (nullable = false)



# Cast Column

In [53]:
df = df.withColumn(
    "monthly_salary",
    col("monthly_salary").cast("double")
)

df = df.withColumn(
    "annual_salary",
    col("annual_salary").cast("double")
)

print("New Schema")
df.printSchema()

New Schema
root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- monthly_salary: double (nullable = false)
 |-- Experience: integer (nullable = true)
 |-- JoiningDate: date (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Department: string (nullable = true)
 |-- Remote: string (nullable = true)
 |-- annual_salary: double (nullable = false)



In [54]:
df.show()

+-------+---+--------+------+--------------+----------+-----------+------+-----------+------+-------------+
|   Name|Age|Category|Region|monthly_salary|Experience|JoiningDate|Rating| Department|Remote|annual_salary|
+-------+---+--------+------+--------------+----------+-----------+------+-----------+------+-------------+
|  Alice| 25|      IT| North|       50000.0|         2| 2022-03-15|   4.2|Engineering|   Yes|     600000.0|
|    Bob| 30|      HR| South|       60000.0|         5| 2019-07-01|   3.8|Recruitment|    No|     720000.0|
|Charlie| 35|      IT|  East|       70000.0|        10| 2014-01-20|   4.7|Engineering|   Yes|     840000.0|
|  David| 28| Finance|  West|       55000.0|         4| 2020-06-10|  NULL| Accounting|    No|     660000.0|
|    Eva| 22|      IT| North|       45000.0|         1| 2023-09-01|   3.5|Engineering|   Yes|     540000.0|
|  Frank| 40|      HR| South|       80000.0|        15| 2009-04-22|   4.9|Recruitment|    No|     960000.0|
|  Grace| 29| Finance|  East

In [56]:
df = df.fillna({"rating":0})

In [59]:
df.count()

40

# Filter Data

In [60]:
filtered_df = df.filter(col("monthly_salary") > 50000)

In [63]:
filtered_df.count()

31

In [64]:

filtered_df.show(5)

+-------+---+--------+------+--------------+----------+-----------+------+-----------+------+-------------+
|   Name|Age|Category|Region|monthly_salary|Experience|JoiningDate|Rating| Department|Remote|annual_salary|
+-------+---+--------+------+--------------+----------+-----------+------+-----------+------+-------------+
|    Bob| 30|      HR| South|       60000.0|         5| 2019-07-01|   3.8|Recruitment|    No|     720000.0|
|Charlie| 35|      IT|  East|       70000.0|        10| 2014-01-20|   4.7|Engineering|   Yes|     840000.0|
|  David| 28| Finance|  West|       55000.0|         4| 2020-06-10|   0.0| Accounting|    No|     660000.0|
|  Frank| 40|      HR| South|       80000.0|        15| 2009-04-22|   4.9|Recruitment|    No|     960000.0|
|  Grace| 29| Finance|  East|       62000.0|         6| 2018-11-05|   4.1| Accounting|   Yes|     744000.0|
+-------+---+--------+------+--------------+----------+-----------+------+-----------+------+-------------+
only showing top 5 rows


In [66]:
result = filtered_df.select(
   
    "name",
    "department",
    "category",
    "monthly_salary",
    "annual_salary"
)

In [69]:
result.show()

+-------+-----------+--------+--------------+-------------+
|   name| department|category|monthly_salary|annual_salary|
+-------+-----------+--------+--------------+-------------+
|    Bob|Recruitment|      HR|       60000.0|     720000.0|
|Charlie|Engineering|      IT|       70000.0|     840000.0|
|  David| Accounting| Finance|       55000.0|     660000.0|
|  Frank|Recruitment|      HR|       80000.0|     960000.0|
|  Grace| Accounting| Finance|       62000.0|     744000.0|
|  Henry|Engineering|      IT|       72000.0|     864000.0|
|    Ivy|Recruitment|      HR|       58000.0|     696000.0|
|   Jack| Accounting| Finance|       67000.0|     804000.0|
|  Karen|Engineering|      IT|       53000.0|     636000.0|
|    Leo| Accounting| Finance|       90000.0|    1080000.0|
|   Paul|Recruitment|      HR|       74000.0|     888000.0|
|  Quinn|Engineering|      IT|       63000.0|     756000.0|
| Rachel| Accounting| Finance|       69000.0|     828000.0|
|    Sam|Recruitment|      HR|       560

In [71]:
result.count()

31

# Save Output

## Saved the processed data in both CSV and Parquet formats.

In [81]:
result.toPandas().to_csv("output/csv_output/csv_output.csv", index=False)

In [83]:
result.toPandas().to_parquet("output/parquet_output/parquet_output.parquet", index=False)

# Read Parquet File

## Verify the saved Parquet File

In [89]:
parquet_df = spark.read.parquet(
    "output/parquet_output/parquet_output.parquet"
)

print("Reading Parquet")
parquet_df.show(31)

Reading Parquet
+-------+-----------+--------+--------------+-------------+
|   name| department|category|monthly_salary|annual_salary|
+-------+-----------+--------+--------------+-------------+
|    Bob|Recruitment|      HR|       60000.0|     720000.0|
|Charlie|Engineering|      IT|       70000.0|     840000.0|
|  David| Accounting| Finance|       55000.0|     660000.0|
|  Frank|Recruitment|      HR|       80000.0|     960000.0|
|  Grace| Accounting| Finance|       62000.0|     744000.0|
|  Henry|Engineering|      IT|       72000.0|     864000.0|
|    Ivy|Recruitment|      HR|       58000.0|     696000.0|
|   Jack| Accounting| Finance|       67000.0|     804000.0|
|  Karen|Engineering|      IT|       53000.0|     636000.0|
|    Leo| Accounting| Finance|       90000.0|    1080000.0|
|   Paul|Recruitment|      HR|       74000.0|     888000.0|
|  Quinn|Engineering|      IT|       63000.0|     756000.0|
| Rachel| Accounting| Finance|       69000.0|     828000.0|
|    Sam|Recruitment|   

In [91]:
spark.stop()